# 06 — Avaliação reproduzível de respostas

Este caderno cria uma pequena bateria de casos e mede uma regra objetiva: a resposta contém o resultado esperado. Ele serve para comparar versões do mesmo modelo; para um projeto real, substitua os casos por um conjunto de teste que não participou do treino.


In [ ]:
from pathlib import Path
import json

ARQUIVO = Path('data/avaliacao_basica.jsonl')
casos = [
    {'prompt': 'Quanto é 6 vezes 8?', 'esperado': '48'},
    {'prompt': 'Quanto é 25% de 120?', 'esperado': '30'},
    {'prompt': 'Qual é o próximo número: 2, 4, 8, 16?', 'esperado': '32'},
]
ARQUIVO.parent.mkdir(exist_ok=True)
with ARQUIVO.open('w', encoding='utf-8') as arquivo:
    for caso in casos:
        arquivo.write(json.dumps(caso, ensure_ascii=False) + '\n')
print(f'{len(casos)} casos salvos em {ARQUIVO}')


## Carregar o modelo

Este caderno é independente: ele carrega o artefato gerado no caderno 01. Para avaliar GGUF ou uma API, substitua apenas a função `gerar`.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_DIR = Path('artifacts/modelo_puro')
if not (MODEL_DIR / 'config.json').is_file():
    raise FileNotFoundError('Modelo ausente. Execute primeiro o caderno 01 até a célula de salvamento.')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR).to(device).eval()

def gerar(prompt: str) -> str:
    entrada = tokenizer(prompt, return_tensors='pt').to(device)
    restante = model.config.n_positions - entrada['input_ids'].shape[-1]
    if restante <= 0:
        return ''
    with torch.no_grad():
        saida = model.generate(**entrada, max_new_tokens=min(48, restante), do_sample=False,
                               pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(saida[0, entrada['input_ids'].shape[-1]:], skip_special_tokens=True)


In [ ]:
resultados = []
for caso in casos:
    resposta = gerar(caso['prompt'])
    acertou = caso['esperado'] in resposta
    resultados.append({**caso, 'resposta': resposta, 'acertou': acertou})

acertos = sum(item['acertou'] for item in resultados)
print(f'Acurácia: {acertos}/{len(resultados)} = {acertos / len(resultados):.1%}')
for item in resultados:
    print(('✓' if item['acertou'] else '✗'), item['prompt'], '=>', repr(item['resposta']))


## Exercício

Amplie a bateria, mantenha-a versionada e compare a acurácia antes e depois de uma alteração. Leia também as respostas: uma métrica simples não substitui inspeção qualitativa.